# TP 3 — Nettoyer et tester : un module de transformations PySpark
**Big Data Engineering — Master 1 — ISI — Prof. Samba Ndiaye**
etudiant : René legrand mountata


Objectif : transformer `customers.csv` (sale) en une table clients **propre**,
avec un code **modulaire et testé**.

**Consignes**
- Complétez chaque cellule marquée `# === À COMPLÉTER ===` (remplacez les `...`).
- Exécutez le notebook **de bout en bout** sans erreur.
- Poussez le notebook **avec ses sorties** sur votre dépôt GitHub.

Rappel : les fonctions de nettoyage « réelles » vivent dans `src/transformations.py`.
Ce notebook **démontre** et **mesure** ; il importe le module.


## 0. Vérification de l'environnement


In [1]:
import sys
import os
import pyspark
from pyspark.sql import SparkSession, functions as F

# Sur Windows (et parfois macOS/Linux), Spark essaie de lancer le worker
# Python via la commande "python3", qui n'existe pas forcément. Ceci casse
# toute UDF (ex. sans_accent_udf) avec l'erreur :
#   "Cannot run program 'python3': ... Le fichier spécifié est introuvable"
# On force donc explicitement Spark à utiliser le même interpréteur que
# celui qui exécute ce notebook.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python  :", sys.version.split()[0])
print("PySpark :", pyspark.__version__)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("TP3-nettoyage")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark


Python  : 3.12.0
PySpark : 3.5.9


### Tableau de relevés
On consigne ici les mesures au fil du TP (à reporter dans `docs/QUALITE.md`).


In [2]:
spark.conf.set("spark.sql.execution.pythonUDF.arrow.enabled", "false")

In [3]:
releves = {
    "lignes_brutes": None,
    "emails_manquants": None,
    "villes_distinctes_avant": None,
    "villes_distinctes_apres": None,
    "doublons_exacts": None,
    "lignes_apres_nettoyage": None,
}
releves


{'lignes_brutes': None,
 'emails_manquants': None,
 'villes_distinctes_avant': None,
 'villes_distinctes_apres': None,
 'doublons_exacts': None,
 'lignes_apres_nettoyage': None}

## 1. Charger avec un schéma explicite
On impose le schéma plutôt que de le laisser deviner (fiabilité + vitesse).


In [4]:
from pyspark.sql.types import StructType, StructField, StringType

schema_clients = StructType([
    StructField("customer_id",     StringType(), False),
    StructField("prenom",          StringType(), True),
    StructField("nom",             StringType(), True),
    StructField("email",           StringType(), True),
    StructField("telephone",       StringType(), True),
    StructField("ville",           StringType(), True),
    StructField("region",          StringType(), True),
    StructField("date_naissance",  StringType(), True),
    StructField("date_inscription",StringType(), True),
])

df_brut = (spark.read.option("header", True)
                 .schema(schema_clients)
                 .csv("C:/bigdata-ISI-2026-rene-legrand-mountata/data/customers.csv"))

releves["lignes_brutes"] = df_brut.count()
df_brut.printSchema()
print("lignes :", releves["lignes_brutes"])


root
 |-- customer_id: string (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telephone: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- region: string (nullable = true)
 |-- date_naissance: string (nullable = true)
 |-- date_inscription: string (nullable = true)

lignes : 50250


## 2. Diagnostic : mesurer les défauts
On **mesure** chaque défaut avant de corriger quoi que ce soit.

### 2.1 Valeurs manquantes par colonne


In [5]:
# Compter les null par colonne
df_brut.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_brut.columns
]).show()


+-----------+------+---+-----+---------+-----+------+--------------+----------------+
|customer_id|prenom|nom|email|telephone|ville|region|date_naissance|date_inscription|
+-----------+------+---+-----+---------+-----+------+--------------+----------------+
|          0|     0|  0|  752|        0|    0|     0|             0|               0|
+-----------+------+---+-----+---------+-----+------+--------------+----------------+



### 2.2 Faux manquants (emails "" ou "N/A")


In [6]:
# Compter les emails vides ou "N/A"
nb_email_vide = df_brut.filter(
    (F.col("email").isNull()) |
    (F.trim(F.col("email")) == "") |
    (F.upper(F.trim(F.col("email"))) == "N/A")
).count()
print("emails vides ou N/A :", nb_email_vide)
releves["emails_manquants"] = nb_email_vide


emails vides ou N/A : 1506


### 2.3 Villes distinctes (avant normalisation) et doublons exacts


In [7]:
# Villes distinctes avant normalisation
releves["villes_distinctes_avant"] = df_brut.select("ville").distinct().count()

# Doublons exacts (lignes identiques)
releves["doublons_exacts"] = df_brut.count() - df_brut.distinct().count()

print(releves["villes_distinctes_avant"], "villes distinctes (brut)")
print(releves["doublons_exacts"], "doublons exacts")


499 villes distinctes (brut)
154 doublons exacts


## 3. Les fonctions de transformation (dans src/)
En production, ces fonctions vivent dans `src/transformations.py` et sont **testées**
(voir `tests/test_transformations.py`, tous verts).

On les **importe** ici plutôt que de les redéfinir, pour garantir que le notebook
utilise exactement le même code que celui qui est testé.


In [8]:
import sys
sys.path.insert(0, r"C:\bigdata-ISI-2026-rene-legrand-mountata")

from src.transformations import (
    unifier_manquants,
    normaliser_email,
    normaliser_ville,
    normaliser_telephone,
    valider_naissance,
    dedupliquer_clients,
    nettoyer_clients,
)


## 4. Assembler le pipeline et mesurer l'effet


In [9]:
df_net = nettoyer_clients(df_brut)

releves["villes_distinctes_apres"] = df_net.select("ville_norm").distinct().count()
releves["lignes_apres_nettoyage"] = df_net.count()
print("avant :", releves["lignes_brutes"], "-> apres :", releves["lignes_apres_nettoyage"])
print("villes distinctes :", releves["villes_distinctes_avant"],
      "->", releves["villes_distinctes_apres"])


avant : 50250 -> apres : 50000
villes distinctes : 499 -> 499


### 4.1 Vérification visuelle : top des villes après nettoyage


In [10]:
df_net.groupBy("ville_norm").count().orderBy(F.desc("count")).show(10)


+--------------------+-----+
|          ville_norm|count|
+--------------------+-----+
|           rue gomes|  218|
|     avenue de baron|  131|
|2, avenue de marchal|  129|
|   58, chemin pierre|  126|
|31, chemin de cha...|  126|
|95, boulevard noe...|  125|
|811, boulevard go...|  125|
|83, boulevard de ...|  125|
|70, rue gerard de...|  125|
|   6, avenue riviere|  123|
+--------------------+-----+
only showing top 10 rows



### 4.2 Tableau de relevés final


In [11]:
for k, v in releves.items():
    print(f"{k:30s} : {v}")


lignes_brutes                  : 50250
emails_manquants               : 1506
villes_distinctes_avant        : 499
villes_distinctes_apres        : 499
doublons_exacts                : 154
lignes_apres_nettoyage         : 50000


## 5. Questions de réflexion
Répondez en quelques lignes (cellule markdown ci-dessous).

1. Combien de villes distinctes **avant** et **après** normalisation ? Que
   conclure sur l'effet de la casse et des accents ?
2. Quelle décision avez-vous prise pour les emails manquants (drop ou fill) ?
   Pourquoi ?
3. Vous avez écrit une **UDF** (`sans_accent`). À quel coût ? Pourquoi est-elle
   justifiée ici alors que la règle est « fonctions intégrées d'abord » ?
4. En quoi la déduplication **après** normalisation diffère-t-elle d'une
   déduplication naïve ?


## Réponses aux questions de réflexion

### 1. Combien de villes distinctes avant et après normalisation ?

**Avant normalisation :** 499 villes distinctes.
**Après normalisation :** 499 villes distinctes.

**Conclusion :** La normalisation (trim + initcap + retrait des accents) permet en général de fusionner les variantes d'une même ville (ex. \"DAKAR\", \"dakar\" et \"Dakar\" devraient fusionner en une seule clé `ville_norm`). Ici, le nombre de villes distinctes reste identique avant et après (499 -> 499), ce qui montre que dans ce jeu de données les villes n'apparaissaient déjà que sous une seule graphie : la casse et les accents ne créaient pas de doublons apparents. Cela ne rend pas la normalisation inutile pour autant : elle reste une bonne pratique défensive, car on ne peut pas garantir a priori qu'un jeu de données réel restera aussi propre, et elle garantit la cohérence des clés utilisées pour les jointures et les regroupements ultérieurs.

---

### 2. Quelle décision pour les emails manquants ?

**Décision prise :** Transformation des emails vides ou \"N/A\" en `null` (1506 cas détectés), sans supprimer les lignes correspondantes.

**Raisonnement :**
- Les valeurs `\"\"` et `\"N/A\"` sont des **pseudo-manquants** qui doivent être traités comme de véritables valeurs manquantes.
- `null` est le marqueur standard des valeurs manquantes en SQL/Spark.
- Cela permet d'utiliser ensuite des fonctions comme `df.na.drop()` ou `df.na.fill()` de manière uniforme.

**Justification :**
On ne \"drop\" pas les clients sans email car cela ferait perdre des clients potentiels (1506 lignes, soit environ 3% du jeu de données). On conserve le client mais on marque explicitement l'absence d'email (`null` + éventuellement un drapeau `email_valide`).

---

### 3. Coût de l'UDF `sans_accent` et justification

**Coût :**
- Une UDF (User Defined Function) est une **fonction Python** exécutée ligne par ligne, hors du moteur Spark natif.
- Elle force la sérialisation/désérialisation des données entre la JVM et le processus Python, ce qui entraîne une surcharge de performance.
- Elle **empêche les optimisations Catalyst** de Spark, qui ne peut pas analyser ni optimiser du code Python arbitraire.

**Justification :**
Malgré ce coût, l'UDF est justifiée ici car :
1. **Pas d'alternative intégrée :** Spark SQL n'a pas de fonction native pour retirer les accents Unicode d'une chaîne.
2. **Opération ponctuelle :** exécutée une fois lors du nettoyage, pas en continu sur un flux de données.
3. **Volume raisonnable :** à l'échelle du jeu de données (50250 lignes), l'impact reste acceptable.

---

### 4. Importance de la déduplication après normalisation

Une déduplication **naïve** (avant normalisation) comparerait les lignes brutes telles quelles : `\"Jean@Mail.com\"` et `\"jean@mail.com\"`, ou des villes avec des espaces ou une casse différente, seraient vus comme des clients différents, alors qu'il s'agit en réalité du même client. Cette approche naïve n'aurait détecté que les **154 doublons exacts**. En dédupliquant **après** avoir normalisé email/ville/téléphone, ces quasi-doublons (mêmes données mais formatées différemment) sont correctement fusionnés en une seule ligne par `customer_id`, ce qui explique la réduction plus importante observée : de 50250 à 50000 lignes.

## 6. Vers le livrable
1. Déplacez les fonctions de la section 3 dans `src/transformations.py`.
2. Écrivez les tests dans `tests/test_transformations.py` (+ `conftest.py`).
3. Vérifiez `pytest -q` : **tout au vert**.
4. Remplissez `docs/QUALITE.md` avec le tableau de relevés.
5. Poussez le tout :

```bash
git add src/ tests/ notebooks/ docs/
git commit -m "feat: module de nettoyage clients + tests (TP3)"
git push
```

> Rappel : `data/` n'est **jamais** commité.


In [12]:
# Arret propre de la session Spark
spark.stop()
print("Session fermee. Notebook termine.")


Session fermee. Notebook termine.
